Download EEG Spectrogram Dataset from Kaggle

In [12]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ayushaote/eeg-spectrogram-images-for-schizophrenia-detection")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'eeg-spectrogram-images-for-schizophrenia-detection' dataset.
Path to dataset files: /kaggle/input/eeg-spectrogram-images-for-schizophrenia-detection


Create Splits for Training, Test, and Val

In [26]:
import os
import shutil
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

data_dir = "/kaggle/input/eeg-spectrogram-images-for-schizophrenia-detection/EEG_IMAGES_VGG16-20250724T160523Z-1-001/EEG_IMAGES_VGG16"
labels_df = pd.read_csv("/kaggle/input/eeg-spectrogram-images-for-schizophrenia-detection/labels.csv")
subject_df = labels_df.groupby("subject_id").first().reset_index()
split_dir = "SPLIT_IMAGES"

# Stratified splitting
train_subjects, test_subjects = train_test_split(
    subject_df,
    test_size=0.15,
    stratify=subject_df["label"],
    random_state=42
)
val_subjects, test_subjects = train_test_split(
    test_subjects,
    test_size=0.15,
    stratify=test_subjects["label"],
    random_state=42
)

# Create a subject_id → split mapping
split_map = {}
for sid in train_subjects["subject_id"]:
    split_map[sid] = "train"
for sid in val_subjects["subject_id"]:
    split_map[sid] = "val"
for sid in test_subjects["subject_id"]:
    split_map[sid] = "test"

# Create folders
for split in ["train", "val", "test"]:
    for label in ["Control", "Patient"]:
        os.makedirs(os.path.join(split_dir, split, label), exist_ok=True)

copied = 0
for _, row in labels_df.iterrows():
    sid = row["subject_id"]
    label = row["label"]
    split = split_map.get(sid)

    #while labels_df lists .edf our images are .png
    original_filename = row["filename"]
    base_filename, _ = os.path.splitext(original_filename)
    image_filename = base_filename + ".png"

    # Construct the full source path to the image file
    src = os.path.join(data_dir, image_filename)
    dst = os.path.join(split_dir, split, label)

    if os.path.exists(src):
        shutil.copy2(src, dst)
        copied += 1
    else:
        print(f"File not found: {src}")
        pass

print(f"Copied {copied} images into split folders at: {split_dir}")

# Set Parameters
img_size = (224, 224)
batch_size = 32

# Data generators
train_gen = ImageDataGenerator(rescale=1./255, zoom_range=0.2, horizontal_flip=True, rotation_range=15).flow_from_directory(
    os.path.join(split_dir, "train"),
    target_size=img_size,
    batch_size=batch_size,
    class_mode="binary"
)

val_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    os.path.join(split_dir, "val"),
    target_size=img_size,
    batch_size=batch_size,
    class_mode="binary"
)

test_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    os.path.join(split_dir, "test"),
    target_size=img_size,
    batch_size=batch_size,
    class_mode="binary",
    shuffle=False
)


Copied 1801 images into split folders at: SPLIT_IMAGES
Found 1537 images belonging to 2 classes.
Found 221 images belonging to 2 classes.
Found 43 images belonging to 2 classes.


CNN

VGG16